In [1]:
import numpy as np
import scipy 
from scipy.io import wavfile
import pandas as pd
import matplotlib.pyplot as plt
# import pyqtgraph as pg
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import gaussian

In [2]:

def wav_info(filepath, print_info):
    """_summary_ Open and read a .wav file, print its details

    Args:
        filepath (_type_): _description_ path to the .wav file

    Returns:
        _type_: _description_ samplerate f_s [Hz], pandas data format (2 channels )
    """
    samplerate, data = wavfile.read(filepath)
    length = data.shape[0] / samplerate
    if print_info == True:
        print(f"number of channels = {data.shape[1]}")
        print(f"playtime = {length:.2f} s")
        print(f"data length = {(data.shape[0])}")
        print(f"samplerate = {samplerate}")
    return samplerate, data


In [3]:
def stft(fs, data, window, hop):
    
    SFT = ShortTimeFFT(window, hop=hop, fs=fs, mfft=200, scale_to='magnitude')
    Sx = SFT.stft(data)  # perform the STFT
    return Sx, SFT

In [4]:
# def plot_result(ch0, ch1, Sx, SFT):

#     N = len(ch0)
#     t_lo, t_hi = SFT.extent(N)[:2]  # time range of plot
#     time_axis = np.linspace(t_lo, t_hi, len(ch0))

#     # Inicjalizacja płótna (Figure) oraz dwóch osi (Axes) w układzie 1 wiersz, 2 kolumny
#     fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16., 8.))

#     # --- Wykres 1 (Lewa strona): Analiza w domenie czasu ---
#     ax1.plot(time_axis,ch0, ".b-", label="channel left")
#     ax1.plot(time_axis,ch1, ".r-", label="channel right")
#     ax1.grid(True)
#     ax1.set_title("Time analysis of a given signal")
#     ax1.set_xlabel("Time [s]")
#     ax1.set_ylabel("Magnitude")
#     ax1.legend()

#     # --- Wykres 2 (Prawa strona): Transformata STFT ---
#     ax2.set_title(rf"STFT ({SFT.m_num*SFT.T:g}$\,s$ Gaussian window, " +
#                 rf"$\sigma_t={g_std*SFT.T}\,$s)")

#     ax2.set(xlabel=f"Time $t$ in seconds ({SFT.p_num(N)} slices, " +
#                 rf"$\Delta t = {SFT.delta_t:g}\,$s)",
#             ylabel=f"Freq. $f$ in Hz ({SFT.f_pts} bins, " +
#                 rf"$\Delta f = {SFT.delta_f:g}\,$Hz)",
#             xlim=(t_lo, t_hi))

#     im2 = ax2.imshow(abs(Sx), origin='lower', aspect='auto',
#                     extent=SFT.extent(N), cmap='inferno')

#     # Wskazanie osi ax2 dla paska skali kolorów (colorbar) zapobiega jego nałożeniu na całą figurę
#     fig.colorbar(im2, ax=ax2, label="Magnitude $|S_x(t, f)|$")

#     # --- Renderowanie ---
#     # Funkcja tight_layout() normalizuje marginesy pomiędzy podobszarami
#     fig.suptitle("Sample \"compiled\" signal analysis", fontsize=16, fontweight='bold')
#     fig.tight_layout()
#     plt.show()

In [5]:
# fs, data = wav_info("../sample_data/harmonics.wav",False)
# np_data = np.asarray(data)

# ch0 = np_data[:, 0]
# ch1 = np_data[:, 1]
# g_std = 8  # standard deviation for Gaussian window in samples
# window = gaussian(50, std=g_std, sym=True)  # symmetric Gaussian window

# Sx, SFT = stft(fs, ch0, window, 200)
# plot_result(ch0, ch1, Sx, SFT)



In [ ]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg') 
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from scipy import signal
import time
import os
import socket


def open_spectrograph():


    # ==========================================
    # 1. PARAMETRY SYSTEMOWE I FIZYCZNE
    # ==========================================
    SOCKET_PATH = "/tmp/fpga_sim.sock"
    FS = 44100            
    CHUNK_SIZE = 2048     
    NPERSEG = 512         
    HISTORY_LEN = 150     

    BIND_IP = "127.0.0.1"
    PORT = 5005



    # Wyliczanie parametrów czasu dla osi odciętych
    TIME_PER_FRAME = CHUNK_SIZE / FS
    TOTAL_HISTORY_TIME = HISTORY_LEN * TIME_PER_FRAME

    # Wyliczanie liczby prążków dla dyskretnej transformaty Fouriera
    freq_bins = NPERSEG // 2 + 1
    waterfall = np.full((freq_bins, HISTORY_LEN), -100.0)

    # ==========================================
    # 2. INICJALIZACJA INTERFEJSU GRAFICZNEGO
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 7))

    # Rezerwacja dolnej przestrzeni okna na suwaki
    plt.subplots_adjust(bottom=0.2) 

    # Inicjalizacja macierzy pikseli (Początkowy widok od ujemnego czasu historii do zera)
    cax = ax.imshow(waterfall, aspect='auto', origin='lower', cmap='inferno', 
                    vmin=-100, vmax=0, extent=[-TOTAL_HISTORY_TIME, 0, 0, FS/2])

    ax.set_title("Real-Time STFT - Czas Bezwzględny (Płynna Oś)")
    ax.set_ylabel("Częstotliwość [Hz]")
    ax.set_xlabel("Czas bezwzględny [s]")
    fig.colorbar(cax, label="Amplituda [dB]")

    # --- KONFIGURACJA WIDŻETÓW STERUJĄCYCH ---
    ax_slider_min = plt.axes([0.15, 0.1, 0.65, 0.03])
    ax_slider_max = plt.axes([0.15, 0.05, 0.65, 0.03])

    slider_min = Slider(ax_slider_min, 'Min dB', -150, 50, valinit=-100)
    slider_max = Slider(ax_slider_max, 'Max dB', -100, 100, valinit=0)

    def on_slider_change(val):
        """Przerwanie aktualizujące progi obcinania palety kolorów."""
        cax.set_clim(vmin=slider_min.val, vmax=slider_max.val)
        fig.canvas.draw_idle()

    slider_min.on_changed(on_slider_change)
    slider_max.on_changed(on_slider_change)

    plt.show(block=False)

    # ==========================================
    # 3. ALOKACJA ZASOBÓW IPC (UNIX DOMAIN SOCKETS)
    # ==========================================
    # if os.path.exists(SOCKET_PATH):
    #     os.remove(SOCKET_PATH)

    # sock = socket.socket(socket.AF_UNIX, socket.SOCK_DGRAM)
    # sock.bind(SOCKET_PATH) 
    # sock.setblocking(False)

    # NEW ---
    import sys
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    try:
        sock.bind((BIND_IP, PORT))
        sock.setblocking(False)
        print(f"[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na {BIND_IP}:{PORT}")
    except OSError as e:
        print(f"[BŁĄD KRYTYCZNY] Nie można zbindować gniazda. Port może być zablokowany: {e}")
        sys.exit(1)

    # end NEW ---


    print(f"[SYSTEM] Zarejestrowano wskaźnik IPC: {SOCKET_PATH}")
    # ==========================================
    # 4. PĘTLA WYKONAWCZA CZASU RZECZYWISTEGO
    # ==========================================
    total_frames = 0
    last_draw_time = time.time()
    DRAW_INTERVAL = 1.0 / 30.0  # Limit odświeżania GUI do 30 FPS

    try:
        while True:
            processed_any = False
            
            # Drenaż bufora jądra - przetwarzanie wszystkich napływających ramek
            while True:
                try:
                    data = sock.recv(CHUNK_SIZE * 4) 
                    
                    # Dekodowanie i obliczanie STFT dla KAŻDEJ ramki
                    chunk = np.frombuffer(data, dtype=np.float32)
                    f_bins, t_bins, Zxx = signal.stft(chunk, fs=FS, nperseg=NPERSEG)
                    slice_idx = Zxx.shape[1] // 2
                    magnitude_db = 20 * np.log10(np.abs(Zxx[:, slice_idx]) + 1e-9) 
                    
                    # Aktualizacja macierzy pikseli w pamięci
                    waterfall = np.roll(waterfall, shift=-1, axis=1)
                    waterfall[:, -1] = magnitude_db
                    
                    total_frames += 1
                    processed_any = True
                    
                except BlockingIOError:
                    break # Brak danych w buforze IPC
                    
            # --- ZARZĄDZANIE WIZUALIZACJĄ (DECIMATION) ---
            if processed_any:
                current_time = time.time()
                
                # Odrysowanie tylko jeśli minął interwał czasu (odciążenie procesora)
                if (current_time - last_draw_time) >= DRAW_INTERVAL:
                    cax.set_data(waterfall)
                    
                    # Logika przesuwania osi czasu na podstawie WSZYSTKICH przetworzonych ramek
                    sim_time = total_frames * TIME_PER_FRAME
                    start_time = sim_time - TOTAL_HISTORY_TIME
                    
                    cax.set_extent([start_time, sim_time, 0, FS/2])
                    ax.set_xlim(start_time, sim_time)
                    
                    fig.canvas.draw_idle()
                    last_draw_time = current_time
            
            # Asynchroniczne przetworzenie zdarzeń (suwaki) niezależnie od odrysowania
            fig.canvas.flush_events()
            
            # Zabezpieczenie przed 100% użyciem CPU w przypadku pustego bufora IPC
            if not processed_any:
                time.sleep(0.005)

    except KeyboardInterrupt:
        print("\n[SYSTEM] Otrzymano sygnał przerwania. Rozpoczynanie zwalniania zasobów...")
    # ==========================================
    # 5. PROCEDURA ZWALNIANIA ZASOBÓW
    # ==========================================
    sock.close()
    if os.path.exists(SOCKET_PATH):
        os.remove(SOCKET_PATH)
    print("[SYSTEM] Zakończono strumieniowanie. Deskryptor pliku został usunięty.")

[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na 127.0.0.1:5005
[SYSTEM] Zarejestrowano wskaźnik IPC: /tmp/fpga_sim.sock

[SYSTEM] Otrzymano sygnał przerwania. Rozpoczynanie zwalniania zasobów...
[SYSTEM] Zakończono strumieniowanie. Deskryptor pliku został usunięty.


In [ ]:
open_spectrograph()

In [15]:
%%bash --bg
# Zapis PID powłoki
echo $$ > /tmp/mic_tx.pid

# Aktywacja środowiska i przejście do katalogu roboczego
source ../bin/activate
cd src
python -u realtime_writer.py

In [1]:
# Odczyt PID, wysłanie sygnału SIGTERM i usunięcie wskaźników
!kill -15 $(cat /tmp/mic_tx.pid) && rm /tmp/mic_tx.pid
!echo "[SYSTEM] Proces zakończony. Logi pozostały w /tmp/mic_tx.log"

/bin/bash: line 1: kill: (122652) - No such process


[SYSTEM] Proces zakończony. Logi pozostały w /tmp/mic_tx.log


In [5]:
import matplotlib
# Wymuszenie systemowego interfejsu graficznego (Tkinter)
matplotlib.use('TkAgg')

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import time
import socket
import sys

# ==========================================
# 1. PARAMETRY SIECIOWE I SYSTEMOWE
# ==========================================
BIND_IP = "127.0.0.1" 
PORT = 5005

FS = 44100            
CHUNK_SIZE = 2048     
NPERSEG = 512         
HISTORY_LEN = 150     

TIME_PER_FRAME = CHUNK_SIZE / FS
TOTAL_HISTORY_TIME = HISTORY_LEN * TIME_PER_FRAME
freq_bins = NPERSEG // 2 + 1
waterfall = np.full((freq_bins, HISTORY_LEN), -100.0)

# ==========================================
# 2. ALOKACJA GNIAZDA UDP (POSIX)
# ==========================================
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

try:
    sock.bind((BIND_IP, PORT))
    sock.setblocking(False)
    print(f"[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na {BIND_IP}:{PORT}")
except OSError as e:
    print(f"[BŁĄD KRYTYCZNY] Nie można zbindować gniazda. Port może być zablokowany: {e}")
    sys.exit(1)

# ==========================================
# 3. INICJALIZACJA ZEWNĘTRZNEGO OKNA
# ==========================================
# Włączenie trybu interaktywnego Matplotlib dla zewnętrznego GUI
plt.ion()

fig, ax = plt.subplots(figsize=(10, 6))
fig.canvas.manager.set_window_title('Hardware-in-the-Loop: STFT Receiver')

cax = ax.imshow(waterfall, aspect='auto', origin='lower', cmap='inferno', 
                vmin=-100, vmax=0, extent=[-TOTAL_HISTORY_TIME, 0, 0, FS/2])

ax.set_title("Real-Time STFT - Zewnętrzny Backend TkAgg")
ax.set_ylabel("Częstotliwość [Hz]")
ax.set_xlabel("Czas bezwzględny [s]")
fig.colorbar(cax, label="Amplituda [dB]")

# Generowanie okna bez blokowania głównego wątku
plt.show(block=False)

# ==========================================
# 4. PĘTLA ODBIORCZA I PRZETWARZANIE
# ==========================================
total_frames = 0
last_draw_time = time.time()
DRAW_INTERVAL = 1.0 / 30.0  # Limit odświeżania okna (~30 FPS)

try:
    # Weryfikacja dostępności okna w systemie operacyjnym
    while plt.fignum_exists(fig.number):
        processed_any = False
        
        # Drenaż bufora systemowego interfejsu sieciowego
        while True:
            try:
                data, addr = sock.recvfrom(CHUNK_SIZE * 4) 
                chunk = np.frombuffer(data, dtype=np.float32)
                
                f_bins, t_bins, Zxx = signal.stft(chunk, fs=FS, nperseg=NPERSEG)
                slice_idx = Zxx.shape[1] // 2
                magnitude_db = 20 * np.log10(np.abs(Zxx[:, slice_idx]) + 1e-9) 
                
                waterfall = np.roll(waterfall, shift=-1, axis=1)
                waterfall[:, -1] = magnitude_db
                
                total_frames += 1
                processed_any = True
                
            except BlockingIOError:
                break 
                
        # Delegacja do biblioteki graficznej API systemu operacyjnego
        if processed_any:
            current_time = time.time()
            if (current_time - last_draw_time) >= DRAW_INTERVAL:
                cax.set_data(waterfall)
                sim_time = total_frames * TIME_PER_FRAME
                start_time = sim_time - TOTAL_HISTORY_TIME
                cax.set_extent([start_time, sim_time, 0, FS/2])
                ax.set_xlim(start_time, sim_time)
                
                fig.canvas.draw_idle()
                last_draw_time = current_time
        
        # Opróżnienie kolejki zdarzeń GUI (np. przesuwanie okna, minimalizacja)
        fig.canvas.flush_events()
        
        if not processed_any:
            time.sleep(0.005)

except KeyboardInterrupt:
    print("\n[SYSTEM] Przerwanie sygnałem z terminala.")
finally:
    sock.close()
    plt.ioff()
    plt.close(fig)
    print("[SYSTEM] Deskryptor gniazda UDP został zwolniony.")

[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na 127.0.0.1:5005
[SYSTEM] Deskryptor gniazda UDP został zwolniony.
